# 13. Interface for warmth and competence prediction

This notebook builds a simple interface that allows a user to upload an artwork image and obtain predicted warmth and competence scores.

The interface uses the best-performing visual pipeline developed in the project. Warmth predictions are the main focus, while competence predictions should be interpreted more cautiously due to weaker model performance in earlier experiments.

In [1]:
!pip install gradio

In [4]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from PIL import Image

import gradio as gr
import torch
from transformers import AutoProcessor, AutoModel
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

## 1. Load training data and prepare the prediction models

The interface uses two separate models:

- a stronger warmth model based on the best-performing filtered setup
- a competence model based on the best available image-only pipeline, although competence predictions should be interpreted more cautiously

In [7]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../Models")

warmth_model = joblib.load(MODEL_DIR / "warmth_rf_siglip_pca50.joblib")
pca_warmth = joblib.load(MODEL_DIR / "warmth_pca50.joblib")
competence_model = joblib.load(MODEL_DIR / "competence_rf_siglip.joblib")

print("Saved models loaded successfully")

Saved models loaded successfully


In [8]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../Models")

warmth_model = joblib.load(MODEL_DIR / "warmth_rf_siglip_pca50.joblib")
pca_warmth = joblib.load(MODEL_DIR / "warmth_pca50.joblib")
competence_model = joblib.load(MODEL_DIR / "competence_rf_siglip.joblib")

print("Saved models loaded successfully")

Saved models loaded successfully


## 2. Load the visual encoder for uploaded images

The interface uses the same SigLIP image encoder to transform an uploaded image into an embedding vector. This embedding is then passed to the saved warmth and competence prediction models.

In [9]:
# ----------------------------
# Load SigLIP encoder
# ----------------------------
MODEL_NAME = "google/siglip-base-patch16-224"

device = torch.device("cpu")
processor = AutoProcessor.from_pretrained(MODEL_NAME)
siglip_model = AutoModel.from_pretrained(MODEL_NAME)
siglip_model.to(device)
siglip_model.eval()

print("SigLIP encoder loaded on:", device)

Loading weights: 100%|██████████| 408/408 [00:00<00:00, 6021.82it/s]

SigLIP encoder loaded on: cpu


In [10]:
def extract_siglip_embedding(image):
    image = image.convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)

    with torch.no_grad():
        vision_outputs = siglip_model.vision_model(pixel_values=pixel_values)

    if hasattr(vision_outputs, "pooler_output") and vision_outputs.pooler_output is not None:
        emb = vision_outputs.pooler_output
    elif hasattr(vision_outputs, "last_hidden_state") and vision_outputs.last_hidden_state is not None:
        emb = vision_outputs.last_hidden_state[:, 0, :]
    else:
        raise ValueError("Unexpected output format from SigLIP vision model")

    emb = emb.detach().cpu().numpy()

    if emb.ndim == 1:
        emb = emb.reshape(1, -1)

    return emb

## 3. Prediction function

This function takes an uploaded image, extracts a SigLIP embedding, and returns predicted warmth and competence scores.

In [11]:
def interpret_score(score, dimension):
    if dimension == "warmth":
        if score >= 0.6:
            return "High warmth"
        elif score >= 0.2:
            return "Moderately warm"
        elif score >= -0.2:
            return "Neutral warmth"
        else:
            return "Low warmth"
    else:  # competence
        if score >= 0.8:
            return "High competence"
        elif score >= 0.5:
            return "Moderate competence"
        elif score >= 0.2:
            return "Low-to-moderate competence"
        else:
            return "Low competence"


def predict_dimensions(image):
    try:
        if image is None:
            return "No image uploaded.", None, None

        emb = extract_siglip_embedding(image)

        # Warmth
        emb_warmth = pca_warmth.transform(emb)
        warmth_pred = float(warmth_model.predict(emb_warmth)[0])

        # Competence
        competence_pred = float(competence_model.predict(emb)[0])

        warmth_text = f"{warmth_pred:.3f}"
        competence_text = f"{competence_pred:.3f}"

        note = "Prediction successful"

        return note, warmth_text, competence_text

    except Exception as e:
        print("ERROR:", e)
        return str(e), "Error", "Error"

In [12]:
def make_scm_plot(items):
    fig, ax = plt.subplots(figsize=(8, 8))

    # SCM axes
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)

    ax.set_xlabel("Warmth")
    ax.set_ylabel("Competence")
    ax.set_title("SCM space")

    # Optional quadrant labels
    ax.text(0.6, 0.9, "High W / High C", ha="center", fontsize=9)
    ax.text(-0.6, 0.9, "Low W / High C", ha="center", fontsize=9)
    ax.text(-0.6, -0.9, "Low W / Low C", ha="center", fontsize=9)
    ax.text(0.6, -0.9, "High W / Low C", ha="center", fontsize=9)

    for item in items:
        x = item["warmth"]
        y = item["competence"]
        img = item["image"].copy().convert("RGB")

        # Make thumbnail
        thumb = img.resize((60, 60))
        imagebox = OffsetImage(np.array(thumb), zoom=0.6)
        ab = AnnotationBbox(imagebox, (x, y), frameon=True, pad=0.2)
        ax.add_artist(ab)

    plt.tight_layout()
    return fig

In [13]:
def add_painting_to_matrix(image, state):
    if image is None:
        empty_fig = make_scm_plot(state if state is not None else [])
        return "No image uploaded.", "", "", empty_fig, state

    if state is None:
        state = []

    emb = extract_siglip_embedding(image)

    # Warmth
    emb_warmth = pca_warmth.transform(emb)
    warmth_pred = float(warmth_model.predict(emb_warmth)[0])

    # Competence
    competence_pred = float(competence_model.predict(emb)[0])

    state.append({
        "image": image.copy(),
        "warmth": warmth_pred,
        "competence": competence_pred
    })

    fig = make_scm_plot(state)

    note = "Painting added to SCM matrix"
    warmth_text = f"{warmth_pred:.3f}"
    competence_text = f"{competence_pred:.3f}"

    return note, warmth_text, competence_text, fig, state

In [14]:
def clear_matrix():
    empty_state = []
    fig = make_scm_plot(empty_state)
    return "", "", "", fig, empty_state

## 4. Launch interface

The interface allows a user to upload an artwork image and receive predicted warmth and competence scores.

In [15]:
with gr.Blocks() as demo:
    gr.Markdown("# Artwork Warmth and Competence Predictor")
    gr.Markdown(
        "Upload an artwork image, predict its warmth and competence, and place it on the SCM matrix."
    )

    state = gr.State([])

    with gr.Row():
        image_input = gr.Image(type="pil", label="Upload an artwork image")

        with gr.Column():
            add_btn = gr.Button("Add painting to matrix")
            clear_btn = gr.Button("Clear matrix")

            note_out = gr.Textbox(label="Note")
            warmth_out = gr.Textbox(label="Predicted Warmth")
            competence_out = gr.Textbox(label="Predicted Competence")

    plot_out = gr.Plot(label="SCM matrix")

    add_btn.click(
        fn=add_painting_to_matrix,
        inputs=[image_input, state],
        outputs=[note_out, warmth_out, competence_out, plot_out, state]
    )

    clear_btn.click(
        fn=clear_matrix,
        inputs=[],
        outputs=[note_out, warmth_out, competence_out, plot_out, state]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
